# X Text Mining for MBG Food Tray in Google Colab

Notebook ini dibuat untuk kebutuhan skripsi:

**Redesign Food Tray MBG pada Sistem Distribusi SPPG**

Target notebook:
1. Mengambil data dari **X API Recent Search**
2. Membersihkan dan menyaring post yang relevan
3. Melakukan **analisis sentimen sederhana**
4. Mengelompokkan isu ke tema desain
5. Mengubah hasilnya menjadi **need statement** untuk desain produk

> Catatan penting:
> - Endpoint ini mengambil post **7 hari terakhir**.
> - Untuk data lebih panjang, lakukan **rolling collection** tiap beberapa hari lalu gabungkan.

In [ ]:
!pip -q install requests pandas openpyxl matplotlib wordcloud

## 1. Isi token dan atur query

Tidak perlu cari satu-satu semua post secara manual.

Strategi yang lebih feasible:
- Buat **4 sampai 8 query cluster**
- Tiap cluster mewakili 1 kelompok isu
- Ambil data dari semua query
- Gabungkan
- Hapus duplikat
- Filter relevansi
- Baru lakukan text mining

Contoh:
- query umum tray MBG
- query suhu dingin / panas
- query tumpah / bocor / tutup
- query material / stainless / food grade
- query distribusi / ikat / stack / handling

In [ ]:
import os
from datetime import datetime, timedelta, timezone

X_BEARER_TOKEN = "ISI_BEARER_TOKEN_X_DI_SINI"
if X_BEARER_TOKEN == "ISI_BEARER_TOKEN_X_DI_SINI":
    X_BEARER_TOKEN = os.getenv("X_BEARER_TOKEN", "")

END_TIME = datetime.now(timezone.utc)
START_TIME = END_TIME - timedelta(days=7)

START_TIME_STR = START_TIME.strftime("%Y-%m-%dT%H:%M:%SZ")
END_TIME_STR = END_TIME.strftime("%Y-%m-%dT%H:%M:%SZ")

print("START:", START_TIME_STR)
print("END  :", END_TIME_STR)

QUERY_PACKS = {
    "umum_tray_mbg": '("food tray MBG" OR "tray MBG" OR "wadah MBG" OR "tempat makan MBG" OR "ompreng MBG") lang:id -is:retweet',
    "isu_suhu": '(("food tray MBG" OR "tray MBG" OR "ompreng MBG" OR "wadah MBG") (dingin OR panas OR hangat OR suhu OR basi OR "cepat basi")) lang:id -is:retweet',
    "isu_tumpah_tutup": '(("food tray MBG" OR "tray MBG" OR "ompreng MBG" OR "wadah MBG") (tumpah OR bocor OR tutup OR rapat OR kuah)) lang:id -is:retweet',
    "isu_material": '(("food tray MBG" OR "tray MBG" OR "ompreng MBG") (stainless OR "ss 304" OR "food grade" OR bahan OR karat)) lang:id -is:retweet',
    "isu_distribusi": '(("food tray MBG" OR "tray MBG" OR "ompreng MBG") (distribusi OR angkut OR ikat OR rafia OR tumpuk OR handling)) lang:id -is:retweet',
    "isu_higiene": '(("food tray MBG" OR "tray MBG" OR "ompreng MBG") (higienis OR sanitasi OR bersih OR kotor OR kontaminasi)) lang:id -is:retweet'
}

MAX_RESULTS_PER_REQUEST = 100
MAX_PAGES_PER_QUERY = 8
MIN_LIKES = 0
MIN_REPLIES = 0
MIN_REPOSTS = 0

## 2. Ambil data dari X API

In [ ]:
import requests
import pandas as pd
import time

SEARCH_URL = "https://api.x.com/2/tweets/search/recent"
HEADERS = {"Authorization": f"Bearer {X_BEARER_TOKEN}"}

TWEET_FIELDS = ",".join([
    "id","text","author_id","created_at","lang","public_metrics","conversation_id","referenced_tweets"
])
USER_FIELDS = ",".join(["id","name","username"])
EXPANSIONS = "author_id"

def fetch_recent_search(query, start_time, end_time, max_results=100, max_pages=5, sleep_sec=1.2):
    all_rows = []
    next_token = None
    page = 0
    while page < max_pages:
        params = {
            "query": query,
            "start_time": start_time,
            "end_time": end_time,
            "max_results": max_results,
            "tweet.fields": TWEET_FIELDS,
            "user.fields": USER_FIELDS,
            "expansions": EXPANSIONS
        }
        if next_token:
            params["next_token"] = next_token

        resp = requests.get(SEARCH_URL, headers=HEADERS, params=params, timeout=60)
        if resp.status_code != 200:
            print("ERROR", resp.status_code, resp.text[:500])
            break

        data = resp.json()
        tweets = data.get("data", [])
        includes = data.get("includes", {})
        users = includes.get("users", [])
        user_map = {u["id"]: u for u in users}

        if not tweets:
            break

        for tw in tweets:
            metrics = tw.get("public_metrics", {})
            author = user_map.get(tw.get("author_id"), {})
            all_rows.append({
                "tweet_id": tw.get("id"),
                "conversation_id": tw.get("conversation_id"),
                "created_at": tw.get("created_at"),
                "author_id": tw.get("author_id"),
                "author_name": author.get("name"),
                "author_username": author.get("username"),
                "lang": tw.get("lang"),
                "text_raw": tw.get("text"),
                "like_count": metrics.get("like_count", 0),
                "reply_count": metrics.get("reply_count", 0),
                "repost_count": metrics.get("retweet_count", 0),
                "quote_count": metrics.get("quote_count", 0),
                "bookmark_count": metrics.get("bookmark_count", 0),
                "query_used": query,
                "tweet_url": f'https://x.com/{author.get("username","unknown")}/status/{tw.get("id")}'
            })

        meta = data.get("meta", {})
        next_token = meta.get("next_token")
        page += 1
        print(f"page {page} | rows so far: {len(all_rows)}")
        if not next_token:
            break
        time.sleep(sleep_sec)

    return pd.DataFrame(all_rows)

In [ ]:
if not X_BEARER_TOKEN:
    raise ValueError("Bearer token X belum diisi.")

all_dfs = []
for label, q in QUERY_PACKS.items():
    print("\n========================")
    print("QUERY LABEL:", label)
    print("QUERY TEXT :", q)
    df_q = fetch_recent_search(
        query=q,
        start_time=START_TIME_STR,
        end_time=END_TIME_STR,
        max_results=MAX_RESULTS_PER_REQUEST,
        max_pages=MAX_PAGES_PER_QUERY
    )
    if not df_q.empty:
        df_q["query_label"] = label
        all_dfs.append(df_q)

if all_dfs:
    raw_df = pd.concat(all_dfs, ignore_index=True)
else:
    raw_df = pd.DataFrame()

print("Total raw rows:", len(raw_df))
raw_df.head()

## 3. Hapus duplikat dan filter engagement minimum

In [ ]:
if raw_df.empty:
    raise ValueError("Data kosong. Cek token, query, atau window waktu.")

df = raw_df.copy()
df = df.drop_duplicates(subset=["tweet_id"]).reset_index(drop=True)
df = df[
    (df["like_count"] >= MIN_LIKES) &
    (df["reply_count"] >= MIN_REPLIES) &
    (df["repost_count"] >= MIN_REPOSTS)
].reset_index(drop=True)

print("Rows after dedupe/filter:", len(df))
df.head()

## 4. Preprocessing teks

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\n|\r|\t", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_clean"] = df["text_raw"].apply(clean_text)
df[["text_raw", "text_clean"]].head(10)

## 5. Filter relevansi

In [ ]:
TRAY_TERMS = ["food tray", "tray", "ompreng", "wadah", "tempat makan", "alat makan"]
MBG_TERMS = ["mbg", "makan bergizi gratis", "sppg"]
ISSUE_TERMS = [
    "dingin", "panas", "hangat", "suhu", "basi", "cepat basi",
    "tumpah", "bocor", "tutup", "rapat", "kuah",
    "stainless", "ss 304", "food grade", "bahan", "karat",
    "higienis", "sanitasi", "kontaminasi", "bersih", "kotor",
    "ikat", "rafia", "tumpuk", "handling", "distribusi"
]

def keyword_hit(text, terms):
    return sum(1 for t in terms if t in text)

def relevance_score(text):
    tray_hit = keyword_hit(text, TRAY_TERMS)
    mbg_hit = keyword_hit(text, MBG_TERMS)
    issue_hit = keyword_hit(text, ISSUE_TERMS)
    return tray_hit * 3 + mbg_hit * 3 + issue_hit * 1

df["score_tray"] = df["text_clean"].apply(lambda x: keyword_hit(x, TRAY_TERMS))
df["score_mbg"] = df["text_clean"].apply(lambda x: keyword_hit(x, MBG_TERMS))
df["score_issue"] = df["text_clean"].apply(lambda x: keyword_hit(x, ISSUE_TERMS))
df["relevance_score"] = df["text_clean"].apply(relevance_score)

relevant_df = df[df["relevance_score"] >= 3].copy().reset_index(drop=True)
print("Relevant rows:", len(relevant_df))
relevant_df[["text_raw", "relevance_score", "score_tray", "score_mbg", "score_issue"]].head(20)

## 6. Analisis sentimen sederhana

In [ ]:
POSITIVE_WORDS = ["bagus", "baik", "aman", "rapi", "kuat", "bersih", "higienis", "sesuai", "mantap", "oke"]
NEGATIVE_WORDS = [
    "dingin", "basi", "tumpah", "bocor", "kotor", "kurang", "buruk", "karat",
    "tidak rapat", "ga rapat", "nggak rapat", "rusak", "berat", "susah", "ribet",
    "takut", "bahaya", "kontaminasi", "jelek", "tipis", "kendala", "masalah"
]

def sentiment_score(text):
    pos = sum(1 for w in POSITIVE_WORDS if w in text)
    neg = sum(1 for w in NEGATIVE_WORDS if w in text)
    return pos - neg

def sentiment_label(score):
    if score > 0:
        return "positif"
    elif score < 0:
        return "negatif"
    return "netral"

relevant_df["sentiment_score"] = relevant_df["text_clean"].apply(sentiment_score)
relevant_df["sentiment_label"] = relevant_df["sentiment_score"].apply(sentiment_label)

relevant_df[["text_raw", "sentiment_score", "sentiment_label"]].head(20)

## 7. Coding tema isu desain

In [ ]:
THEME_RULES = {
    "retensi_suhu": ["dingin", "panas", "hangat", "suhu", "cepat basi", "basi"],
    "kebocoran_tumpah": ["tumpah", "bocor", "kuah", "tidak rapat", "ga rapat", "nggak rapat", "tutup"],
    "material_keamanan": ["stainless", "ss 304", "food grade", "bahan", "karat"],
    "higienitas": ["higienis", "sanitasi", "kotor", "bersih", "kontaminasi"],
    "handling_stackability": ["ikat", "rafia", "tumpuk", "handling", "angkut", "stabil"],
    "ergonomi_penggunaan": ["berat", "susah", "ribet", "nyaman", "mudah dibawa"]
}

def assign_themes(text):
    hit_themes = []
    for theme, keywords in THEME_RULES.items():
        for kw in keywords:
            if kw in text:
                hit_themes.append(theme)
                break
    return hit_themes

relevant_df["themes"] = relevant_df["text_clean"].apply(assign_themes)
relevant_df["n_themes"] = relevant_df["themes"].apply(len)
coded_df = relevant_df[relevant_df["n_themes"] > 0].copy().reset_index(drop=True)
coded_df[["text_raw", "themes"]].head(20)

## 8. Ringkasan tema

In [ ]:
from collections import Counter

theme_counter = Counter()
for lst in coded_df["themes"]:
    theme_counter.update(lst)

theme_summary = pd.DataFrame(
    [{"theme": k, "count": v} for k, v in theme_counter.items()]
).sort_values("count", ascending=False).reset_index(drop=True)

theme_summary

## 9. Kata dominan

In [ ]:
from collections import Counter

STOPWORDS = {
    "dan","di","yang","untuk","ke","dari","ini","itu","pada","dengan","atau","karena",
    "jadi","aja","saja","juga","ga","gak","nggak","nya","ya","kok","nih","sih","mah",
    "the","a","an","is","are","to","of","in","for","on","at","be","as","it",
    "mbg","sppg"
}

def tokenize(text):
    return [w for w in text.split() if len(w) > 2 and w not in STOPWORDS]

all_tokens = []
for t in coded_df["text_clean"]:
    all_tokens.extend(tokenize(t))

word_counts = Counter(all_tokens)
top_words_df = pd.DataFrame(word_counts.most_common(50), columns=["word", "count"])
top_words_df.head(20)

## 10. Kutipan representatif per tema

In [ ]:
def representative_posts_for_theme(df_in, theme, top_n=5):
    df_t = df_in[df_in["themes"].apply(lambda x: theme in x)].copy()
    if df_t.empty:
        return pd.DataFrame()
    df_t["rep_score"] = (
        df_t["like_count"] +
        df_t["reply_count"] * 2 +
        df_t["repost_count"] * 2 +
        df_t["relevance_score"] * 3
    )
    cols = [
        "tweet_id", "created_at", "author_username", "text_raw",
        "like_count", "reply_count", "repost_count", "tweet_url", "rep_score"
    ]
    return df_t.sort_values("rep_score", ascending=False)[cols].head(top_n)

representative_posts = []
for theme in theme_summary["theme"].tolist():
    temp = representative_posts_for_theme(coded_df, theme, top_n=5)
    if not temp.empty:
        temp["theme"] = theme
        representative_posts.append(temp)

representative_posts_df = pd.concat(representative_posts, ignore_index=True) if representative_posts else pd.DataFrame()
representative_posts_df.head(20)

## 11. Ubah tema menjadi need statement

In [ ]:
THEME_TO_NEED = {
    "retensi_suhu": "Food tray harus mampu memperlambat penurunan suhu makanan selama distribusi.",
    "kebocoran_tumpah": "Food tray harus mampu meminimalkan risiko kebocoran dan tumpahan selama pengangkutan dan distribusi.",
    "material_keamanan": "Food tray harus menggunakan material yang aman pangan, mudah dibersihkan, dan sesuai dengan kebutuhan distribusi MBG.",
    "higienitas": "Food tray harus mendukung higienitas dan meminimalkan potensi kontaminasi selama penanganan dan distribusi.",
    "handling_stackability": "Food tray harus mudah ditumpuk, stabil saat dibawa, dan mendukung handling distribusi yang lebih aman.",
    "ergonomi_penggunaan": "Food tray harus mudah digunakan, tidak terlalu berat, dan mendukung kemudahan operasional bagi petugas distribusi."
}

need_rows = []
for _, row in theme_summary.iterrows():
    theme = row["theme"]
    need_rows.append({
        "theme": theme,
        "frequency": int(row["count"]),
        "need_statement": THEME_TO_NEED.get(theme, "")
    })

need_statement_df = pd.DataFrame(need_rows).sort_values("frequency", ascending=False).reset_index(drop=True)
need_statement_df

## 12. Visualisasi

In [ ]:
import matplotlib.pyplot as plt

sent_summary = coded_df["sentiment_label"].value_counts().reset_index()
sent_summary.columns = ["sentiment", "count"]

plt.figure(figsize=(6,4))
plt.bar(sent_summary["sentiment"], sent_summary["count"])
plt.title("Distribusi Sentimen")
plt.xlabel("Sentimen")
plt.ylabel("Jumlah Post")
plt.show()

plt.figure(figsize=(8,4))
plt.bar(theme_summary["theme"], theme_summary["count"])
plt.title("Distribusi Tema Isu")
plt.xlabel("Tema")
plt.ylabel("Jumlah Post")
plt.xticks(rotation=45, ha="right")
plt.show()

## 13. Review manual yang disarankan

In [ ]:
review_cols = [
    "tweet_id", "created_at", "author_username", "text_raw", "text_clean",
    "relevance_score", "sentiment_label", "themes", "tweet_url"
]

manual_review_df = coded_df[review_cols].copy()
manual_review_df["manual_keep"] = ""
manual_review_df["manual_sentiment"] = ""
manual_review_df["manual_theme_1"] = ""
manual_review_df["manual_theme_2"] = ""
manual_review_df["manual_notes"] = ""
manual_review_df.head(20)

## 14. Simpan semua output

In [ ]:
from pathlib import Path

output_dir = Path("/content/output_mbg_x_text_mining")
output_dir.mkdir(parents=True, exist_ok=True)

raw_df.to_csv(output_dir / "01_raw_x_posts.csv", index=False)
df.to_csv(output_dir / "02_cleaned_x_posts.csv", index=False)
relevant_df.to_csv(output_dir / "03_relevant_x_posts.csv", index=False)
coded_df.to_csv(output_dir / "04_coded_x_posts.csv", index=False)
theme_summary.to_csv(output_dir / "05_theme_summary.csv", index=False)
top_words_df.to_csv(output_dir / "06_top_words.csv", index=False)
representative_posts_df.to_csv(output_dir / "07_representative_posts.csv", index=False)
need_statement_df.to_csv(output_dir / "08_need_statements.csv", index=False)
manual_review_df.to_csv(output_dir / "09_manual_review_sheet.csv", index=False)

with pd.ExcelWriter(output_dir / "mbg_x_text_mining_output.xlsx", engine="openpyxl") as writer:
    raw_df.to_excel(writer, sheet_name="raw_x_posts", index=False)
    df.to_excel(writer, sheet_name="cleaned_x_posts", index=False)
    relevant_df.to_excel(writer, sheet_name="relevant_posts", index=False)
    coded_df.to_excel(writer, sheet_name="coded_posts", index=False)
    theme_summary.to_excel(writer, sheet_name="theme_summary", index=False)
    top_words_df.to_excel(writer, sheet_name="top_words", index=False)
    representative_posts_df.to_excel(writer, sheet_name="representative_posts", index=False)
    need_statement_df.to_excel(writer, sheet_name="need_statements", index=False)
    manual_review_df.to_excel(writer, sheet_name="manual_review", index=False)

print("Saved to:", output_dir)
print(list(output_dir.iterdir()))

## 15. Cara pakai hasil ini untuk skripsi

Urutan kerja:
1. Ambil data X dengan query cluster  
2. Bersihkan data  
3. Filter relevansi  
4. Analisis sentimen  
5. Coding tema isu  
6. Ambil kutipan representatif  
7. Ubah tema menjadi need statement  
8. Gabungkan dengan benchmarking  
9. Turunkan ke atribut desain dan spesifikasi  
10. Buat 2 alternatif desain  
11. Validasi dengan expert judgement

Tips:
- Jangan andalkan sentimen saja
- Fokus pada **tema masalah** dan **need statement**
- Lakukan **manual review** agar hasil akademiknya lebih kuat
- Kalau mau data lebih banyak, kumpulkan **rolling** tiap beberapa hari lalu gabungkan